In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine, Integer, String, Date, Boolean, Numeric, Table, Column, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from typing import List, Optional
from datetime import date

In [3]:
engine = create_engine("sqlite:///restaurant_backoffice.db")

class Base(DeclarativeBase):
    ...


In [4]:
class Employee(Base):
    __tablename__ = "employee"
    
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(64), nullable=False)
    surname: Mapped[str] = mapped_column(String(64), nullable=False)
    national_id: Mapped[str] = mapped_column(String(20), nullable=False, unique=True)
    role: Mapped[str] = mapped_column(String(20), nullable=False)

    # Relació 1:1
    credentials: Mapped[Optional["AppCredential"]] = relationship(back_populates="employee", uselist=False)


    # Relació cap a Sale (1:N)
    sales: Mapped[list["Sale"]] = relationship(back_populates="employee")

class AppCredential(Base):
    __tablename__ = "app_credential"
    
    # Foreign Key cap a Employee
    id_employee: Mapped[int] = mapped_column(Integer, ForeignKey("employee.id", ondelete="CASCADE"), primary_key=True)
    
    email: Mapped[str] = mapped_column(String(128), unique=True, nullable=False)
    password_hash: Mapped[str] = mapped_column(String(255), nullable=False)
    
    # Relació inversa cap a employee
    employee: Mapped["Employee"] = relationship(back_populates="credentials")

class Category(Base):
    __tablename__ = "category"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    category_name: Mapped[str] = mapped_column(String(64), nullable=False)

    # Relació: Una categoria té una LLISTA de plats, per això fem servir List["Dish"], significant que és una relació 1:N, en aquest cas estem al costat "One" de la relació. (1 categoria té molts plats)
    dishes: Mapped[list["Dish"]] = relationship(back_populates="category")

# Definició de la taula intermèdia dish_allergen
dish_allergen = Table(
    "dish_allergen",
    Base.metadata,
    Column("id_dish", ForeignKey("dish.id", ondelete="CASCADE"), primary_key=True),
    Column("id_allergen", ForeignKey("allergen.id", ondelete="CASCADE"), primary_key=True),
)

# Definició de la taula intermèdia menu_dish
menu_dish = Table(
    "menu_dish",
    Base.metadata,
    Column("id_menu", ForeignKey("menu.id", ondelete="CASCADE"), primary_key=True),
    Column("id_dish", ForeignKey("dish.id", ondelete="CASCADE"), primary_key=True),
)

class Dish(Base):
    __tablename__ = "dish"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    dish_name: Mapped[str] = mapped_column(String(64), nullable=False)
    description: Mapped[Optional[str]] = mapped_column(String)
    is_active: Mapped[bool] = mapped_column(default=True)
    
    # La Foreign Key sempre va al costat "Many"
    id_category: Mapped[int] = mapped_column(ForeignKey("category.id"), nullable=False)
    
    # Relació N:1 (Many-to-One): Un plat pertany a UNA sola categoria,
    # però una categoria pot tenir molts plats.
    category: Mapped["Category"] = relationship(back_populates="dishes")

    # Relacions Many-to-Many amb Allergen i Menu
    allergens: Mapped[list["Allergen"]] = relationship(
        secondary=dish_allergen, back_populates="dishes"
    )
    menus: Mapped[list["Menu"]] = relationship(
        secondary=menu_dish, back_populates="dishes"
    )

class Allergen(Base):
    __tablename__ = "allergen"

    id: Mapped[int] = mapped_column(primary_key=True)
    allergen_name: Mapped[str] = mapped_column(String(64), nullable=False)

    # Relació Many-to-Many amb Dish
    # 'secondary' indica la taula associativa que connecta Dish i Allergen
    # 'back_populates' indica el nom de l'atribut a l'altra classe que fa referència a aquesta relació
    dishes: Mapped[list["Dish"]] = relationship(
        secondary=dish_allergen, back_populates="allergens"
    )

class Menu(Base):
    __tablename__ = "menu"

    id: Mapped[int] = mapped_column(primary_key=True)
    menu_name: Mapped[str] = mapped_column(String(64), nullable=False)
    price: Mapped[float] = mapped_column(Numeric(4, 2), nullable=False)

    # Relació Many-to-Many amb Dish
    dishes: Mapped[list["Dish"]] = relationship(
        secondary=menu_dish, back_populates="menus"
    )

    # Relació 1:N amb Sale (un menú pot estar en moltes vendes, però una venda només té un menú)
    sales: Mapped[list["Sale"]] = relationship(back_populates="menu")

class Sale(Base):
    __tablename__ = "sale"

    # Clau primària composta per id_menu i id_employee
    id_menu: Mapped[int] = mapped_column(ForeignKey("menu.id"), primary_key=True)
    id_employee: Mapped[int] = mapped_column(ForeignKey("employee.id"), primary_key=True)

    # Atributs adicionals de la relació N:M 
    sale_date: Mapped[date] = mapped_column(Date, primary_key=True) # Afegit a PK si un mateix empleat ven el mateix menú en dies diferents
    quantity_sold: Mapped[int] = mapped_column(Integer, default=1) # Afegin un default per facilitar la creació de vendes quan només es ven una unitat

    # Relació cap a Menu i Employee
    menu: Mapped["Menu"] = relationship(back_populates="sales")
    employee: Mapped["Employee"] = relationship(back_populates="sales")

# 4. Creació de les taules a la base de dades
Base.metadata.create_all(engine)

In [5]:
#Printar base de dades
print(Base.metadata.tables.keys())

dict_keys(['employee', 'app_credential', 'category', 'dish_allergen', 'menu_dish', 'dish', 'allergen', 'menu', 'sale'])
